# Explainable AI across Text and VisionA walkthrough of the rebuilt pipeline. Everything here calls the same `xai` package thebatch runs use — nothing is reimplemented for the demo, so what you see is what thereport is built from.**Contents**1. Setup and configuration2. The three datasets3. Explaining a tweet — LIME, SHAP, prototypes4. Explaining a movie review — where prototypes catch what tokens miss5. Explaining an image — Grad-CAM, SHAP, LIME6. Aggregate results across every prediction

## 1. SetupThe whole pipeline is driven by `config.yaml`: paths, seeds, and the perturbation budgets for each method.

In [ ]:
import sys, json, warningssys.path.insert(0, "../src")warnings.filterwarnings("ignore")from xai import config as cfgmodcfg = cfgmod.load("../config.yaml")device = cfgmod.get_device(cfg.raw["device"])print(f"device: {device}   seed: {cfg.seed}")for k in cfg.task_keys:    t = cfg.task(k)    print(f"  {k:9s} {t.kind:5s}  {t.name}")

## 2. The datasetsAll three are normalised onto one record shape, so nothing downstream branches on which dataset it is looking at.

In [ ]:
from xai.data import describefor k in cfg.task_keys:    print(f"\n=== {k} ===")    print(json.dumps(describe(cfg.task(k)), indent=2)[:700])

## 3. Explaining a tweetThree methods, three different questions:- **LIME** — perturb words, fit a local linear model, read off the coefficients- **SHAP** — each token's average marginal contribution over orderings (Shapley value)- **Prototypes** — which training examples does this sit next to?Token weights are signed toward the **predicted** class throughout.

In [ ]:
from xai.models import load_modelfrom xai.explainers.text import LimeText, ShapText, PrototypeTextfrom xai.data import load_text_splittask = cfg.task("tweet")model = load_model(task, device)tweet = "the new update is such a mess, nothing works anymore"probs = model.predict_proba([tweet])[0]pid = int(probs.argmax())print(f"text: {tweet}")for lab, p in sorted(zip(model.labels, probs), key=lambda x: -x[1]):    print(f"  {lab:<9} {p:.4f}  {'#' * int(p*40)}")print(f"\nprediction: {model.labels[pid]}")

In [ ]:
lime_e = LimeText(model, task, top_k_tokens=10).explain(tweet, pid)shap_e = ShapText(model, task, top_k_tokens=10).explain(tweet, pid)for name, e in (("LIME", lime_e), ("SHAP", shap_e)):    print(f"\n{name}  ({e.runtime_s:.2f}s)  stats={e.stats}")    for t, w in e.tokens:        bar = ("+" if w >= 0 else "-") * min(int(abs(w) * 40) + 1, 40)        print(f"  {t.strip()[:20]:<20} {w:+.4f}  {bar}")

`surrogate_r2` is LIME's own reliability score: how well its local linear model actually reproduces the classifier near this input. A low value means the bar chart above looks authoritative but means very little.

In [ ]:
train = load_text_split(task, "train")proto = PrototypeText(    model, task, [s.text for s in train], [s.label_id for s in train],    k=3, cache_path=cfg.artifact_dir("tweet") / "train_embeddings.npz",)e = proto.explain_batch([tweet], [pid])[0]print(f"neighbour agreement: {e.stats['neighbour_agreement']:.0%}\n")for n in e.neighbours:    tick = "OK " if n["agrees_with_prediction"] else "!! "    print(f"{tick} sim {n['similarity']:.3f}  [{n['label']}]  {n['text'][:90]}")

## 4. A movie review — where prototypes earn their placeThe LoRA model is the weakest of the three (80.4%). This example shows why running morethan one *kind* of explanation matters: the token methods produce noise, while theexample-based method flatly contradicts the prediction.

In [ ]:
mtask = cfg.task("movie")mmodel = load_model(mtask, device)review = ("The premise was promising but the pacing dragged and the dialogue felt "          "wooden. I checked my watch twice.")mprobs = mmodel.predict_proba([review])[0]mpid = int(mprobs.argmax())print(f"prediction: {mmodel.labels[mpid]}  ({mprobs[mpid]:.3f})")mtrain = load_text_split(mtask, "train")mproto = PrototypeText(    mmodel, mtask, [s.text for s in mtrain], [s.label_id for s in mtrain],    k=3, cache_path=cfg.artifact_dir("movie") / "train_embeddings.npz",)me = mproto.explain_batch([review], [mpid])[0]print(f"neighbour agreement: {me.stats['neighbour_agreement']:.0%}\n")for n in me.neighbours:    tick = "OK " if n["agrees_with_prediction"] else "!! "    print(f"{tick} sim {n['similarity']:.3f}  [{n['label']}]  {n['text'][:90]}")

When every retrieved neighbour disagrees with the prediction, the model has made a call its own training data does not support. That is a signal no token-attribution method produces.

## 5. Explaining an imageAll three image methods attribute in **pixel space** — normalisation happens inside themodel's forward pass, not as preprocessing — so the three maps describe the same functionand can be compared directly.

In [ ]:
import numpy as np, matplotlib.pyplot as pltfrom xai.data import load_image_samplesfrom xai.explainers.image import GradCAM, ShapImage, LimeImagegtask = cfg.task("glasses")gmodel = load_model(gtask, device)samples = load_image_samples(gtask)sample = [s for s in samples if s.label == "glasses"][7]x = gmodel.load_tensor(sample.path)p = gmodel.predict_proba_tensor(x.unsqueeze(0))[0]gpid = int(p.argmax())print(f"{sample.sample_id}: true={sample.label}  pred={gmodel.labels[gpid]} ({p[gpid]:.4f})")maps = {}for name, ex in (("gradcam", GradCAM(gmodel, gtask)),                 ("shap",    ShapImage(gmodel, gtask)),                 ("lime",    LimeImage(gmodel, gtask))):    e, m = ex.explain(x, gpid)    maps[name] = m    print(f"  {name:<8} {e.runtime_s:6.2f}s  focus={e.stats.get('top5pct_mass')}  peak={e.stats.get('peak_yx')}")

In [ ]:
img = x.permute(1, 2, 0).cpu().numpy()fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))axes[0].imshow(img); axes[0].set_title(f"input — {sample.label}"); axes[0].axis("off")for ax, (name, m) in zip(axes[1:], maps.items()):    ax.imshow(img)    if name == "gradcam":        ax.imshow(m, cmap="inferno", alpha=0.5)    else:        lim = np.percentile(np.abs(m), 99) or 1e-9        ax.imshow(m, cmap="coolwarm", alpha=0.55, vmin=-lim, vmax=lim)    ax.set_title(name); ax.axis("off")plt.tight_layout(); plt.show()

`top5pct_mass` — the share of total attribution held by the strongest 5% of pixels — is a direct measure of how *focused* a map is. Grad-CAM is bounded by a 7x7 feature grid, so it localises regions; the other two can be sharper but noisier.

## 6. Aggregate resultsThese come from the full runs, where **every** prediction in every evaluation set was explained.

In [ ]:
summary_path = cfg.artifacts / "summary_all.json"if not summary_path.exists():    print("Run  python -m xai.run --all  then  python -m xai.summarize --all  first.")else:    allsum = json.loads(summary_path.read_text())    for k, s in allsum.items():        perf = s["performance"]        print(f"\n=== {s['name']} ===")        print(f"  {s['n_records']} predictions explained | accuracy {perf['accuracy']:.4f} | macro-F1 {perf['macro_f1']:.4f}")        for m, v in s["methods"].items():            extra = f" surrogate R2={v['surrogate_r2_mean']:.3f}" if "surrogate_r2_mean" in v else ""            print(f"    {m:<10} {v['runtime_s_mean']:6.2f}s/sample  "                  f"{v['runtime_s_total']/60:6.1f} min total  failures {v['failure_rate']:.1%}{extra}")        for a in s["agreement"]:            print(f"    agreement: {a}")

## Where to go nextThe two datasets ship human ground truth that this project does not yet use:- `Dataset/Glasses/attention_label/` — human attention masks for all 2,614 images- the tweet dataset's `selected_text` column — human rationale spans for every tweetScoring the generated explanations against those (pointing game and IoU for images,token-level F1 for text) would turn this comparison into a benchmark against humanjudgement. That is the single most valuable extension to this work.